# Waste detection engine: train, build, benchmark

Runs the whole project on a Colab or Kaggle GPU: fine-tunes the detector on TACO,
builds the C++/CUDA engine with the TensorRT FP16 backend, and benchmarks it
against the plain Python pipeline.

**Before running**

- Turn the GPU on (Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4).
- On Kaggle, also turn Internet on.
- Give the notebook the images, because the image host refuses bulk downloads from these services:
  - **Kaggle:** upload `taco_images.zip` as a dataset and add it to this notebook with *Add input*.
  - **Colab:** put `taco_images.zip` in the top level of your Google Drive. The notebook asks to connect Drive.
  - Or set `DATA_ZIP` below to a path or a direct download link.

Then choose *Run all*.

In [ ]:
BRANCH = "cuda-inference-engine"
EPOCHS = 60          # lower this if the session is short on time
DATA_ZIP = ""        # optional: path or URL of taco_images.zip; empty = look in the usual places

## 1. Get the code and check the GPU

In [ ]:
import os, subprocess

ON_KAGGLE = os.path.isdir("/kaggle/working")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = os.path.join(ROOT, "MAJOR-PROJECT")

if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/venkat2912/MAJOR-PROJECT.git", REPO], check=True)
os.chdir(REPO)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU found. Turn the GPU on in the notebook settings and run again."
print("GPU:", gpu.stdout.strip())
print("Working in", os.getcwd())

## 2. Bring in the images

In [ ]:
import glob, shutil, urllib.request, zipfile

DEST = os.path.join(REPO, "data", "taco")
os.makedirs(DEST, exist_ok=True)


def count_images():
    return len(glob.glob(os.path.join(DEST, "images", "*", "*.jpg")))


def unpack(zip_path):
    print("unpacking", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DEST)


def find_source():
    """Returns ('zip', path), ('dir', path of a folder holding images/train) or None."""
    if DATA_ZIP.startswith("http"):
        target = os.path.join(ROOT, "taco_images.zip")
        if not os.path.exists(target):
            print("downloading", DATA_ZIP)
            urllib.request.urlretrieve(DATA_ZIP, target)
        return "zip", target
    if DATA_ZIP:
        return "zip", DATA_ZIP
    if ON_KAGGLE:
        # Kaggle unpacks uploaded archives, so look for the folder first.
        for d in glob.glob("/kaggle/input/**/images/train", recursive=True):
            return "dir", os.path.dirname(os.path.dirname(d))
        for z in glob.glob("/kaggle/input/**/taco_images.zip", recursive=True):
            return "zip", z
        return None
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        z = "/content/drive/MyDrive/taco_images.zip"
        if os.path.exists(z):
            return "zip", z
    except Exception as e:
        print("Google Drive not available:", e)
    return None


if count_images() == 0:
    source = find_source()
    if source is None:
        print("No taco_images.zip found. The training step will try the image host directly,")
        print("which usually refuses most requests from Colab and Kaggle.")
    elif source[0] == "zip":
        unpack(source[1])
    else:
        print("copying from", source[1])
        shutil.copytree(os.path.join(source[1], "images"), os.path.join(DEST, "images"),
                        dirs_exist_ok=True)

print(count_images(), "images ready")

## 3. Train

Images already present are skipped; any that are missing are tried from the image
host for at most five minutes. Each photo is also cut into the tiles the engine
uses. This is the long step.

In [ ]:
!EPOCHS={EPOCHS} MAX_WAIT=300 MIN_FRACTION=0 bash scripts/run_all.sh train

## 4. Build the engine and benchmark

Builds a TensorRT FP16 engine from the trained model, compiles the C++/CUDA
engine and compares it with the Python baseline on a validation photo.

In [ ]:
!bash scripts/run_all.sh trt

## 5. Collect the results

In [ ]:
import shutil
from IPython.display import Image, display

RESULTS = os.path.join(ROOT, "waste_results")
os.makedirs(RESULTS, exist_ok=True)
wanted = ["models/waste.pt", "models/waste.names", "runs/waste/results.csv",
          "runs/waste/results.png", "runs/waste/confusion_matrix.png",
          "out/engine_tensorrt.json", "out/engine_torchscript.json", "out/baseline.json",
          "out/annotated_tensorrt.jpg"]
for f in wanted:
    if os.path.exists(f):
        shutil.copy(f, RESULTS)
    else:
        print("missing:", f)
archive = shutil.make_archive(RESULTS, "zip", RESULTS)
print("saved", archive)

for f in ("runs/waste/results.png", "out/annotated_tensorrt.jpg"):
    if os.path.exists(f):
        display(Image(filename=f, width=900))

if not ON_KAGGLE:
    from google.colab import files
    files.download(archive)